Import alignment results of protein fasta to proteome

In [1]:
from collections import namedtuple
from pathlib import Path
import pandas as pd
from Bio import SeqIO
from pysam import VariantFile
import edlib
import re


# path
path_work_dir = Path("/home/b05b01002/HDD/project_scRNAed")
path_output = path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/"
path_peptide_identification_result = path_work_dir /  "external_data/align_to_proteome/20250709_RNA_editing_Ptr_6 frame longest ORF_PeptideGroups.txt"
# path_edited_protein_annotated_non_syn = path_work_dir / "tests/test-VariantAnnotation/edited_protein_annotated_frame_nonsynonymous.fa"
# path_edited_protein_annotated_syn = path_work_dir / "tests/test-VariantAnnotation/edited_protein_annotated_frame_synonymous.fa"
# path_edited_protein_6frames = path_work_dir / "tests/test-VariantAnnotation/edited_protein_6frames.fa"
# path_original_protein_6frames = path_work_dir / "tests/test-VariantAnnotation/original_protein_6frames.fa"
# path_variant_annotation = path_work_dir / "tests/test-VariantAnnotation2/variant_annotation_hom_ref.csv"
path_exons_gtf = path_work_dir / "tests/test-VariantAnnotation2/exons_gtf.csv"
path_exons_mod_gtf = path_work_dir / "tests/test-VariantAnnotation2/exons_mod_gtf.csv"
path_variant_hom_vcf = path_work_dir / "tests/test-VariantAnnotation2/ptr_tenx_batch1_hom.vcf.bgz"
path_original_exons_fasta = path_work_dir / "tests/test-VariantAnnotation2/original_exons.fa"
path_edited_exons_fasta = path_work_dir / "tests/test-VariantAnnotation2/edited_exons.fa"
path_original_protein_fasta = path_work_dir / "tests/test-VariantAnnotation2/original_protein.fa"
path_edited_protein_fasta = path_work_dir / "tests/test-VariantAnnotation2/edited_protein.fa"
path_annotation_info = path_work_dir / "references/Ptr/annotation/Ptrichocarpa_533_v4.1.annotation_info.txt"

In [ ]:
# load
peptide_identification_result = pd.read_csv(path_peptide_identification_result, sep="\t")[["Sequence", "Protein Accessions"]].drop_duplicates()
# edited_protein_annotated_non_syn = SeqIO.to_dict(SeqIO.parse(path_edited_protein_annotated_non_syn, "fasta"))
# edited_protein_annotated_syn = SeqIO.to_dict(SeqIO.parse(path_edited_protein_annotated_syn, "fasta"))
# edited_protein_annotated = edited_protein_annotated_non_syn | edited_protein_annotated_syn
# edited_protein_6frames = SeqIO.to_dict(SeqIO.parse(path_edited_protein_6frames, "fasta"))
# original_protein_6frames = SeqIO.to_dict(SeqIO.parse(path_original_protein_6frames, "fasta"))
# variant_annotation = pd.read_csv(path_variant_annotation, sep=",")
exons_gtf = pd.read_csv(path_exons_gtf, sep=",")
exons_mod_gtf = pd.read_csv(path_exons_mod_gtf, sep=",")
variant_hom_vcf = VariantFile(path_variant_hom_vcf)
original_exons_fasta = SeqIO.to_dict(SeqIO.parse(path_original_exons_fasta, "fasta"))
edited_exons_fasta = SeqIO.to_dict(SeqIO.parse(path_edited_exons_fasta, "fasta"))
original_protein_fasta = SeqIO.to_dict(SeqIO.parse(path_original_protein_fasta, "fasta"))
edited_protein_fasta = SeqIO.to_dict(SeqIO.parse(path_edited_protein_fasta, "fasta"))


In [586]:
def parse_cigar(cigar_str):
    # Match one or more digits followed by a single character (e.g., "10=", "2X")
    return [(int(length), op) for length, op in re.findall(r"(\d+)([=XID])", cigar_str)]

def get_mismatch_pos(parsed_cigar):
    expanded_cigar = "".join([o*n for n, o in parsed_cigar])
    return [i.span() for i in re.finditer(r"([XID]{1,})", expanded_cigar)]

def mismatch_pos_0based(query, target, mode):
    res = edlib.align(query, target, mode, "path")
    parsed_cigar = parse_cigar(res["cigar"])
    return get_mismatch_pos(parsed_cigar)

def compare_list_of_ranges(list1, list2):
    pos1 = set()
    pos2 = set()
    for start, end in list1:
        pos1 = pos1 | set([i for i in range(start, end)])
    for start, end in list2:
        pos2 = pos2 | set([i for i in range(start, end)])
    overlap = pos1.intersection(pos2)
    return overlap


Annotated variant on protein translated from non-canonical ORF

In [414]:
modifications = exons_mod_gtf.apply(
    lambda x: [rec for rec in variant_hom_vcf.fetch(x.seqnames, x.start, x.end)], 
    1
)
exons_mod_gtf["modifications"] = modifications

In [ ]:
DEBUG = True
# named tuple
VariantAnnotation = namedtuple("VariantAnnotation", [
    "CHROM",
    "POS",
    "REF",
    "ALT",
    "GENEID",
    "FRAME",
    "PROTEINLOC",
    "CONSEQUENCE",
    "REFCODON",
    "VARCODON",
    "REFAA",
    "VARAA"
])
# variant annotation
variant_annotations = []
for idx, rows in exons_mod_gtf.iterrows():
    # skip row if no modification was found on this exon
    if len(rows.modifications) == 0:
        continue

    # 1. get transcript sequence
    transcript = original_exons_fasta[rows.group_name]
    transcript_edited = edited_exons_fasta[rows.group_name]
    # 2. get gtf of the exons
    gtf = exons_mod_gtf[exons_mod_gtf["group_name"] == rows.group_name]
    # 3. get the base-by-base coordinate on chromosome
    coord = []
    for idx, exon_range in gtf.iterrows():
        if exon_range.strand == "-":
            coord += [*range(exon_range.start, exon_range.end + 1)][::-1]
        else:
            coord += [*range(exon_range.start, exon_range.end + 1)]
    # for each modification
    for modification in rows.modifications:
        t_loc = coord.index(modification.pos)
        for frame in [1, 2, 3, -1, -2, -3]:
            offset = abs(frame) - 1
            temp_transcript = transcript
            temp_transcript_edited = transcript_edited
            temp_t_loc = t_loc
            if frame < 0:
                temp_transcript = temp_transcript.reverse_complement()
                temp_transcript_edited = temp_transcript_edited.reverse_complement()
                temp_t_loc = len(temp_transcript) - 1 - t_loc
            # translate
            # protein = temp[offset:].translate()
            # protein_edited = temp_edited[offset:].translate()
            protein = original_protein_fasta[f"{rows.group_name}_frame={frame}"]
            protein_edited = edited_protein_fasta[f"{rows.group_name}_frame={frame}"]
            
            # codon
            t_loc_codon_1 = temp_t_loc - ((temp_t_loc - offset) % 3)
            codon_ref = temp_transcript[t_loc_codon_1:t_loc_codon_1+3]
            codon_var = temp_transcript_edited[t_loc_codon_1:t_loc_codon_1+3]
            if len(codon_ref) < 3:
                print(modification, frame, "no-translatable")
                continue
            
            # protein
            aa_ref = codon_ref.translate()
            aa_var = codon_var.translate()
            
            # get protein location
            # TODO: fix this
            p_loc = t_loc_codon_1 // 3
            if DEBUG == True:
                a, b = temp_transcript[temp_t_loc], temp_transcript_edited[temp_t_loc]
                c, d = modification.id, frame
                e, f = aa_ref.seq, protein[p_loc] # equal
                g, h = aa_var.seq, protein_edited[p_loc] # equal
                print()

            
            # get consequence
            def get_consequence(aa_ref, aa_var):
                if aa_ref == aa_var:
                    return "synonymous"
                else:
                    if aa_ref == "*":
                        return "nonstop"
                    if aa_var == "*":
                        return "nonsense"
                    return "missense"
    
            # record to list
            va = VariantAnnotation(
                CHROM=modification.chrom,
                POS=modification.pos,
                REF=modification.ref,
                ALT=modification.alts[0],
                GENEID=rows.group_name,
                FRAME=frame,
                PROTEINLOC=p_loc,
                CONSEQUENCE=get_consequence(str(aa_ref.seq), str(aa_var.seq)),
                REFCODON=str(codon_ref.seq),
                VARCODON=str(codon_var.seq),
                REFAA=str(aa_ref.seq),
                VARAA=str(aa_var.seq)
            )           
            variant_annotations.append(va)

df = pd.DataFrame(variant_annotations)
df.to_csv(
    path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames.tsv",
    index=False,
    sep="\t"
)

The input protein fasta

In [458]:
peptide_identification_result = pd.read_csv(path_peptide_identification_result, sep="\t")[["Sequence", "Protein Accessions"]].drop_duplicates()
peptide_identification_result = peptide_identification_result.assign(
    protein_frame=peptide_identification_result["Protein Accessions"].str.split(";")
).explode("protein_frame").reset_index(drop=True)
peptide_identification_result["protein_frame"] = peptide_identification_result["protein_frame"].str.strip()
peptide_identification_result.head()

,Sequence,Protein Accessions,protein_frame
0,VNNEAVQK,Potri.013G061800.6_frame=1;Potri.013G061800.8_...,Potri.013G061800.6_frame=1
1,VNNEAVQK,Potri.013G061800.6_frame=1;Potri.013G061800.8_...,Potri.013G061800.8_frame=3
2,VNNEAVQK,Potri.013G061800.6_frame=1;Potri.013G061800.8_...,Potri.013G061800.2_frame=2
3,VNNEAVQK,Potri.013G061800.6_frame=1;Potri.013G061800.8_...,Potri.013G061800.10_frame=1
4,VNNEAVQK,Potri.013G061800.6_frame=1;Potri.013G061800.8_...,Potri.013G061800.12_frame=2


In [ ]:
variant_annotations = pd.read_csv(
    path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames.tsv",
    sep="\t"
)
variant_annotations.index = variant_annotations.apply(lambda x: f"{x.GENEID}_frame={x.FRAME}", 1)
variant_annotations

In [ ]:
variant_annotations.loc[:, "PEPTIDE_COVERED"] = None
variant_annotations.loc[:, "PEPTIDE_SEQ"] = None
variant_annotations.loc[:, "PEPTIDE_LOC"] = None
for pf, partial_df in peptide_identification_result.groupby("protein_frame"):
    edited_protein = edited_protein_fasta[pf].seq
    if not pf in variant_annotations.index:
        continue
    variant_on_pf = variant_annotations.loc[pf,:]
    variant_p_loc = variant_on_pf["PROTEINLOC"].tolist()
    if not isinstance(variant_p_loc, list):
        variant_p_loc = [variant_p_loc]
    variant_interval = [(i, i+1) for i in variant_p_loc]
    peptide_locations = []
    for seq in partial_df["Sequence"]:
        res = edlib.align(
            query=seq,
            target=edited_protein,
            mode="HW",
            task="locations",
            k=0
        )
        peptide_locations += res["locations"]
    overlaps = compare_list_of_ranges(peptide_locations, variant_interval)
    if "nonsense" in variant_on_pf["CONSEQUENCE"]:
        break
    if len(overlaps) > 0:
        mask = variant_annotations["PROTEINLOC"].isin(overlaps)
        variant_annotations.loc[mask & (variant_annotations.index == pf), "PEPTIDE_COVERED"] = True
        variant_annotations.loc[mask & (variant_annotations.index == pf), "PEPTIDE_SEQ"] =  ",".join(partial_df["Sequence"].tolist())
        variant_annotations.loc[mask & (variant_annotations.index == pf), "PEPTIDE_LOC"] = ",".join([f"{x}-{y}" for x, y in peptide_locations])

In [627]:
variant_annotations.to_csv(
    path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames_peptide_validation.tsv",
    index=False,
    sep="\t"
)

Match _Populus_ Gene ID with validated RNA editing sites to _Arabidopsis_ Gene ID

In [59]:
annotation_info = pd.read_csv(
    path_annotation_info,
    sep="\t"
)
annotation_info.index = annotation_info["transcriptName"]
variant_annotation_w_validation = pd.read_csv(
    path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames_peptide_validation.tsv",
    sep="\t"
)
variant_annotation_w_validation["GID"] = variant_annotation_w_validation["GENEID"].str.split(".").apply(lambda x: f"{x[0]}.{x[1]}")
variant_annotation_w_validation.head()


,CHROM,POS,REF,ALT,GENEID,FRAME,PROTEINLOC,CONSEQUENCE,REFCODON,VARCODON,REFAA,VARAA,PEPTIDE_COVERED,PEPTIDE_SEQ,PEPTIDE_LOC,PEPTIDE_SEQS,PEPTIDE_LOCS,GID
0,Chr01,111038,T,C,Potri.001G001400.2,1,295,missense,TTA,TCA,L,S,NaN,NaN,NaN,NaN,NaN,Potri.001G001400
1,Chr01,111038,T,C,Potri.001G001400.2,2,295,missense,TAT,CAT,Y,H,NaN,NaN,NaN,NaN,NaN,Potri.001G001400
2,Chr01,111038,T,C,Potri.001G001400.2,3,294,synonymous,CTT,CTC,L,L,NaN,NaN,NaN,NaN,NaN,Potri.001G001400
3,Chr01,111038,T,C,Potri.001G001400.2,-1,610,synonymous,TAA,TGA,*,*,NaN,NaN,NaN,NaN,NaN,Potri.001G001400
4,Chr01,111038,T,C,Potri.001G001400.2,-2,610,missense,AAG,GAG,K,E,NaN,NaN,NaN,NaN,NaN,Potri.001G001400


In [84]:
is_covered = variant_annotation_w_validation["PEPTIDE_COVERED"] == True
not_synonymous = variant_annotation_w_validation["CONSEQUENCE"] != "synonymous"
is_synonymous = variant_annotation_w_validation["CONSEQUENCE"] == "synonymous"
variant_annotation_w_validation.loc[is_covered & is_synonymous,].apply(lambda x: "%s:%s" % (x.CHROM, x.POS), 1).unique().__len__()

299

In [102]:
is_reversed_gene = variant_annotation_w_validation.loc[is_covered,].groupby("GENEID").apply(lambda x: all(x["FRAME"] < 0))
reversed_gene = is_reversed_gene[is_reversed_gene].index

In [ ]:
annotation_info.loc[reversed_gene,]

,#pacId,locusName,transcriptName,peptideName,Pfam,Panther,KOG,ec,KO,GO,best_arabi_gene,best_arabi_defline
GENEID,,,,,,,,,,,,
Potri.001G028600.1,PAC:42793211,Potri.001G028600,Potri.001G028600.1,Potri.001G028600.1.p,PF00195,PTHR11877 PTHR11877:SF27,EC:2.3.1.74,NaN,K00660,GO:0003824 GO:0008152 GO:0009058 GO:0016747,NaN,NaN
Potri.005G062800.1,PAC:42803611,Potri.005G062800,Potri.005G062800.1,Potri.005G062800.1.p,PF00005,PTHR24220,EC:3.6.3.25,KOG2355,NaN,GO:0005524 GO:0016887,AT4G33460,ABC transporter family protein
Potri.007G081100.1,PAC:42765562,Potri.007G081100,Potri.007G081100.1,Potri.007G081100.1.p,PF00646 PF01344,PTHR24414 PTHR24414:SF37,NaN,NaN,NaN,GO:0005515,AT2G21950,SKP1 interacting partner 6
Potri.010G031250.1,PAC:42797405,Potri.010G031250,Potri.010G031250.1,Potri.010G031250.1.p,NaN,PTHR15117,NaN,NaN,NaN,NaN,NaN,NaN
Potri.010G031250.2,PAC:42797406,Potri.010G031250,Potri.010G031250.2,Potri.010G031250.2.p,NaN,PTHR15117,NaN,NaN,NaN,NaN,NaN,NaN
Potri.010G031250.3,PAC:42797407,Potri.010G031250,Potri.010G031250.3,Potri.010G031250.3.p,NaN,PTHR15117,NaN,NaN,NaN,NaN,NaN,NaN
Potri.010G031250.4,PAC:42797408,Potri.010G031250,Potri.010G031250.4,Potri.010G031250.4.p,NaN,PTHR15117,NaN,NaN,NaN,NaN,NaN,NaN
Potri.010G031250.5,PAC:42797409,Potri.010G031250,Potri.010G031250.5,Potri.010G031250.5.p,NaN,PTHR15117,NaN,NaN,NaN,NaN,NaN,NaN
